# Stream Analysis — Ground Truth Pipeline

This notebook runs Part 2 of the document image analysis pipeline.
**Run the [overview](stream-analysis-overview-demo.ipynb) notebook first** — this part loads
cached outputs from Part 1, and uses the same `image_dir`/`output_dir`.

Steps:
1. Cluster-stratified sampling
2. Export to Label Studio
3. Active learning (after you have annotated some images)

The active-learning step needs real manual annotation work in Label Studio first, so it can't
be baked into this demo's outputs the way the other steps are -- it gracefully reports what's
missing instead of failing if you run this notebook as-is without annotating anything first.

In [1]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')  # keep notebook outputs tidy

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(levelname)-8s  %(message)s', datefmt='%H:%M:%S')

## Configuration

Use the same `image_dir` and `output_dir` as in the overview notebook.

In [2]:
from archival_structures.stream_analysis import AnalysisConfig

cfg = AnalysisConfig(
    image_dir='../../data/thumbs/NL-AsnDA/NL-AsnDA_0114.11/NL-AsnDA_0114.11_1',
    output_dir='../../archival_structures/stream_analysis/outputs/demo',
    samples_per_cluster=30,
)

# Or load from YAML:
# cfg = AnalysisConfig.from_yaml('archival_structures/stream_analysis/config.yaml')

## Step 1: Stratified Sampling + Label Studio Export

In [3]:
from archival_structures.stream_analysis import run_export

export_path = run_export(cfg)
print(f'\nLabel Studio import file: {export_path}')

16:32:04  INFO      Loading cached embeddings from disk


16:32:04  INFO      Loaded 630 embeddings of dimension 768


16:32:04  INFO      Loading cached layout features from disk


16:32:04  INFO      Loading cached UMAP projection from ../../archival_structures/stream_analysis/outputs/demo/umap_projection.json


16:32:04  INFO      Loading cached clustering results from ../../archival_structures/stream_analysis/outputs/demo/clustering_results.json


...

16:32:04  INFO      Total sampled: 209 images across 9 clusters


16:32:04  INFO      Sample saved → ../../archival_structures/stream_analysis/outputs/demo/groundtruth/sampled_ids.json


16:32:04  INFO      === Exporting to Label Studio ===


16:32:04  INFO      Label Studio export: 209 tasks (0 with VLM pre-annotations) → ../../archival_structures/stream_analysis/outputs/demo/groundtruth/label_studio_import.json



Done. 209 images exported to ../../archival_structures/stream_analysis/outputs/demo/groundtruth/label_studio_import.json
  0 images have VLM pre-annotations.

Next steps:
  1. Run print_label_studio_config() to get the XML for Label Studio.
  2. Import the JSON file into Label Studio and start annotating.
  3. Export labels and save to: ../../archival_structures/stream_analysis/outputs/demo/groundtruth/labels.json
  4. Run run_active_learning() to get suggestions for the next batch.

Label Studio import file: ../../archival_structures/stream_analysis/outputs/demo/groundtruth/label_studio_import.json


## Step 2: Print Label Studio XML Config

Paste this XML into Label Studio → Settings → Labelling Interface.

In [4]:
from archival_structures.stream_analysis.groundtruth.label_studio_export import print_label_config

print_label_config()

<View>
  <Image name="image" value="$image" zoom="true"/>

  <Choices name="document_type" toName="image" choice="single" showInLine="true">
    <Header value="Document Type"/>
    <Choice value="letter"/>
    <Choice value="form"/>
    <Choice value="table"/>
    <Choice value="photograph"/>
    <Choice value="map"/>
    <Choice value="certificate"/>
    <Choice value="invoice"/>
    <Choice value="register"/>
    <Choice value="newspaper"/>
    <Choice value="handwritten_note"/>
    <Choice value="printed_text"/>
    <Choice value="mixed"/>
    <Choice value="other"/>
  </Choices>

  <Choices name="writing_mode" toName="image" choice="single" showInLine="true">
    <Header value="Writing Mode"/>
    <Choice value="handwritten"/>
    <Choice value="typed"/>
    <Choice value="printed"/>
    <Choice value="mixed"/>
    <Choice value="none"/>
  </Choices>

  <Choices name="colour_profile" toName="image" choice="single" showInLine="true">
    <Header value="Colour Profile"/>
    <Choice 

## Step 3: Active Learning

Run this cell **after** annotating images in Label Studio and saving labels to `cfg.label_file`.

Expected label file format:
```text
{"path/to/image.jp2": "handwritten_letter", ...}
```

In [5]:
from pathlib import Path

if Path(cfg.label_file).exists():
    from archival_structures.stream_analysis import run_active_learning

    suggestions = run_active_learning(cfg)

    print(f'\nTop suggestions to annotate next:')
    for s in suggestions[:10]:
        print(f"  [{s['uncertainty']:.3f}] {s['image_id']}  →  {s['top_prediction']} ({s['top_prob']:.0%})")
else:
    print(f"No label file at {cfg.label_file} yet -- this step needs at least 10 manually "
          "annotated images (across 2+ classes) from Label Studio, saved to that path, before "
          "active learning can train a classifier and suggest what to label next.")

No label file at ../../archival_structures/stream_analysis/outputs/demo/groundtruth/labels.json yet -- this step needs at least 10 manually annotated images (across 2+ classes) from Label Studio, saved to that path, before active learning can train a classifier and suggest what to label next.


## Manual exploration

You can also work with the individual components directly:

In [6]:
from archival_structures.stream_analysis.overview.embeddings import extract_embeddings
from archival_structures.stream_analysis.overview.clustering import run_clustering, get_cluster_members
from archival_structures.stream_analysis.overview.layout_analysis import extract_all_layout_features
from archival_structures.stream_analysis.groundtruth.stratified_sampling import stratified_sample

config = cfg.to_dict()
embeddings, image_ids = extract_embeddings(config)
layout_features = extract_all_layout_features(config)
clustering = run_clustering(embeddings, image_ids, config, layout_features)
cluster_members = get_cluster_members(clustering)

print(f"Total images: {len(image_ids)}")
print(f"Clusters: {clustering['n_clusters']}")

# Inspect a specific cluster
cluster_id = 0
print(f"\nCluster {cluster_id} ({len(cluster_members[cluster_id])} images):")
for img_id in cluster_members[cluster_id][:5]:
    print(f"  {img_id}")

16:32:05  INFO      Loading cached embeddings from disk


16:32:05  INFO      Loaded 630 embeddings of dimension 768


16:32:05  INFO      Loading cached layout features from disk


16:32:05  INFO      Loading cached UMAP projection from ../../archival_structures/stream_analysis/outputs/demo/umap_projection.json


16:32:05  INFO      Loading cached clustering results from ../../archival_structures/stream_analysis/outputs/demo/clustering_results.json


Total images: 630
Clusters: 8

Cluster 0 (29 images):
  ../../data/thumbs/NL-AsnDA/NL-AsnDA_0114.11/NL-AsnDA_0114.11_1/thumb-width_300-scan-NL-AsnDA_0114.11_1_0028.jp2.png
  ../../data/thumbs/NL-AsnDA/NL-AsnDA_0114.11/NL-AsnDA_0114.11_1/thumb-width_300-scan-NL-AsnDA_0114.11_1_0067.jp2.png
  ../../data/thumbs/NL-AsnDA/NL-AsnDA_0114.11/NL-AsnDA_0114.11_1/thumb-width_300-scan-NL-AsnDA_0114.11_1_0097.jp2.png
  ../../data/thumbs/NL-AsnDA/NL-AsnDA_0114.11/NL-AsnDA_0114.11_1/thumb-width_300-scan-NL-AsnDA_0114.11_1_0099.jp2.png
  ../../data/thumbs/NL-AsnDA/NL-AsnDA_0114.11/NL-AsnDA_0114.11_1/thumb-width_300-scan-NL-AsnDA_0114.11_1_0142.jp2.png
